# 2.4 — The Transformations to Avoid

**Chapter 2, section 2.6.1** (*`reduceByKey` versus `groupByKey`*), and
**chapter 5**, where the same file is used to watch the shuffle in the Spark UI.

**The question this notebook answers:** two operators give the identical answer. What is
different, how much is it, and how do you see it?

The chapter's claim is precise and it is measurable:

> Both operators therefore shuffle, but `groupByKey` shuffles the raw data whereas
> `reduceByKey` shuffles a summary. On a large data set this is not a marginal difference; it
> is the difference between a job that completes and a job that exhausts its memory.

So this notebook does not argue. It reads the shuffle sizes out of Spark's own metrics, on
data large enough for the difference to be real, and then reproduces the failure mode that
makes it matter — a single oversized key.

Covers **Exercise 5** (shuffle volume, and what skew does to it), **Exercise 7** (the four
poor operator choices), and **Exercise 10** (build a skewed RDD, time both operators, and read
the task-duration distribution of the shuffle stage). Chapter 5's exercise is the same thing
at twenty million records: change `N` below.

Runs on a laptop in about a minute.

## Setup

The measurements come from Spark's own metrics, read from the **local** Spark UI of this very
session over its REST API. That is the same data the Jobs and Stages tabs display; reading it
programmatically just means the numbers survive into the saved notebook instead of vanishing
with the web page. Nothing outside this machine is contacted.

In [1]:
# --- CS-777 session setup ------------------------------------------------
# Chapter 2, section 2.6.
import os, json, time, random, tempfile, urllib.request
from pyspark.sql import SparkSession

SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-2.4")
         .master("local[*]")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

sc = spark.sparkContext
UI = sc.uiWebUrl
APP = json.load(urllib.request.urlopen(UI + "/api/v1/applications"))[0]["id"]

def _get(path):
    return json.load(urllib.request.urlopen(f"{UI}/api/v1/applications/{APP}{path}"))

def stages_of(group):
    """Every stage Spark ran under a named job group, in order.

    Scoping by job group rather than by stage name matters: several cells in this
    notebook call groupByKey, and a name-based lookup would silently add them together.
    """
    stage_ids = sorted({sid for j in _get("/jobs")
                        if j.get("jobGroup") == group
                        for sid in j["stageIds"]})
    return [_get(f"/stages/{sid}/0") for sid in stage_ids]

def task_ms(stage):
    """Min / median / max task run time in a stage, in milliseconds."""
    try:
        q = _get(f"/stages/{stage['stageId']}/0/taskSummary?quantiles=0,0.5,1.0")
        return q["executorRunTime"]
    except Exception:
        return None

def report(group):
    """The two numbers that matter: bytes written into the shuffle, and how evenly the
    work on the far side of it was spread."""
    st = stages_of(group)
    write = max(st, key=lambda s: s["shuffleWriteBytes"])
    read = max(st, key=lambda s: s["shuffleReadBytes"])
    q = task_ms(read) or [0, 0, 0]
    return {"write_bytes": write["shuffleWriteBytes"],
            "write_records": write["shuffleWriteRecords"],
            "task_min": q[0], "task_median": q[1], "task_max": q[2],
            "straggler": q[2] / max(q[1], 1)}

def timed(label, fn):
    sc.setJobGroup(label, label)
    t = time.perf_counter()
    out = fn()
    return out, time.perf_counter() - t

print("Spark", spark.version, "on", sc.master)
# UI holds a URL containing this machine's hostname or LAN address, so only the
# port is printed -- open http://localhost:<port> to follow along in the browser.
print("Spark UI: http://localhost:%s" % UI.rsplit(":", 1)[1])

Spark 4.2.0 on local[*]
Spark UI: http://localhost:4040


## 1. The warm-up: the same answer, twice

A pair RDD of six records. Both operators produce the total per key.

In [2]:
rdd2 = sc.parallelize([('the', 1), ('the', 1), ('the', 2),
                       ('for', 1), ('for', 2), ('for', 1)])

print("groupByKey, raw          :", [(k, list(v)) for k, v in rdd2.groupByKey().collect()])
print("groupByKey + mapValues   :", sorted(rdd2.groupByKey().mapValues(sum).collect()))
print("reduceByKey              :", sorted(rdd2.reduceByKey(lambda a, b: a + b).collect()))

assert (sorted(rdd2.groupByKey().mapValues(sum).collect())
        == sorted(rdd2.reduceByKey(lambda a, b: a + b).collect()))
print("\nidentical answers -- verified, not asserted in prose")

groupByKey, raw          : [('for', [1, 2, 1]), ('the', [1, 1, 2])]
groupByKey + mapValues   : [('for', 4), ('the', 4)]
reduceByKey              : [('for', 4), ('the', 4)]



identical answers -- verified, not asserted in prose


Two things about that cell are worth naming before moving on.

`groupByKey()` returns an iterable of values per key, not a list — printing it needs an
explicit `list(v)`, which is a small hint that the values are being *held* rather than
combined.

And it is `mapValues(sum)`, not `map(lambda kv: sum(kv[1]))`. `mapValues` tells Spark the keys
are untouched, so an existing partitioning by key stays valid and a further shuffle can be
avoided. That is **Exercise 7(a)**, and section 6 returns to it.

## 2. The measurement

Six records will not show a cost difference; eight million will. Two details of the generator
below are deliberate.

The records are built **on the executors**, with `sc.range(...).mapPartitionsWithIndex(...)`,
rather than as a Python list passed to `parallelize`. A list of eight million tuples would have
to be built in the driver's memory and shipped out, which is slow and is exactly the pattern
section 2.3.2 of the chapter warns `parallelize` is unsuitable for.

The values are **random floats with a fixed seed per partition**. Values that compress well
would hide the thing being measured, because Spark compresses shuffle data by default; and the
seed makes the run reproducible.

In [3]:
N = 8_000_000        # chapter 5's exercise is this same notebook with N = 20_000_000
KEYS = 50

def make_balanced(idx, it):
    rnd = random.Random(1000 + idx)
    for _ in it:
        yield (f"k{rnd.randrange(KEYS):03d}", rnd.random())

data = sc.range(0, N, numSlices=8).mapPartitionsWithIndex(make_balanced).cache()
print(f"{data.count():,} records, {KEYS} keys, {data.getNumPartitions()} partitions")

8,000,000 records, 50 keys, 8 partitions


In [4]:
gbk_answer, gbk_secs = timed("gbk", lambda: sorted(data.groupByKey().mapValues(sum).collect()))
rbk_answer, rbk_secs = timed("rbk", lambda: sorted(data.reduceByKey(lambda a, b: a + b).collect()))

# Same answer.  This is the premise of the whole comparison, so it is checked, not claimed.
assert len(gbk_answer) == len(rbk_answer) == KEYS
for (k1, v1), (k2, v2) in zip(gbk_answer, rbk_answer):
    assert k1 == k2 and abs(v1 - v2) < 1e-6
print(f"both operators returned the same {KEYS} totals\n")

gbk, rbk = report("gbk"), report("rbk")

print(f"{'':14s}{'shuffle write':>17s}{'records':>10s}{'seconds':>9s}")
print(f"{'groupByKey':14s}{gbk['write_bytes']:>15,} B{gbk['write_records']:>10,}{gbk_secs:>9.2f}")
print(f"{'reduceByKey':14s}{rbk['write_bytes']:>15,} B{rbk['write_records']:>10,}{rbk_secs:>9.2f}")
print(f"\ngroupByKey shuffled {gbk['write_bytes'] / rbk['write_bytes']:,.0f}x more DATA "
      f"for the same answer.")

both operators returned the same 50 totals

                  shuffle write   records  seconds
groupByKey         72,079,773 B        64     0.33
reduceByKey            10,072 B        64     0.23

groupByKey shuffled 7,156x more DATA for the same answer.


## 3. Reading that table honestly

The `records` column is the same for both operators, and that deserves an explanation, because
the chapter's figure shows `groupByKey` shipping six records where `reduceByKey` ships three.

That figure describes the model, and it is what Spark does in Scala. **PySpark's
`groupByKey` does group locally within each partition first** — it builds one list per key per
partition and ships that. So the number of records crossing the network is the same in both
cases; what differs is what each record *contains*. `reduceByKey` sends one number per key per
partition. `groupByKey` sends every value that key had, packed into a list.

That is why the honest column here is **bytes**, not records, and it is why the ratio above is
in the hundreds rather than exactly the ratio of values-per-key. The chapter's conclusion is
unchanged and the mechanism is the one it names: one operator ships a summary, the other ships
the data.

> Chapter 5 quotes Karau and colleagues: a `groupByKey` whose shuffle read was 86 MB against a
> 200 MB input, and the same computation with `reduceByKey` reading a few hundred kilobytes.
> That is the same measurement at a larger scale, and the shape matches what is above.

## 4. Why it matters: one oversized key

The size of the shuffle is the ordinary cost. The failure is different, and it is the one
Exercise 10 asks you to reproduce.

`groupByKey` produces, for each key, an iterator over **all** of that key's values, and that
collection cannot itself be distributed — it must fit in memory on the single executor that
owns the key. So `groupByKey` does not fail because the data is large. It fails because **one
key** is large, and a data set that would otherwise fit comfortably is enough to do it.

Below, one key holds 90 % of the records.

In [5]:
def make_skewed(idx, it):
    rnd = random.Random(2000 + idx)
    for _ in it:
        key = "HOT" if rnd.random() < 0.9 else f"k{rnd.randrange(KEYS):03d}"
        yield (key, rnd.random())

skewed = sc.range(0, N, numSlices=8).mapPartitionsWithIndex(make_skewed).cache()
print(f"{skewed.count():,} records")
for k, c in sorted(skewed.countByKey().items(), key=lambda kv: -kv[1])[:4]:
    print(f"  {k:6s} {c:>10,}  ({100 * c / N:5.1f}%)")

8,000,000 records
  HOT     7,199,971  ( 90.0%)
  k004       16,349  (  0.2%)
  k049       16,206  (  0.2%)
  k039       16,190  (  0.2%)


In [6]:
_, sk_gbk_secs = timed("skew-gbk", lambda: skewed.groupByKey().mapValues(sum).collect())
_, sk_rbk_secs = timed("skew-rbk", lambda: skewed.reduceByKey(lambda a, b: a + b).collect())

sk_gbk, sk_rbk = report("skew-gbk"), report("skew-rbk")

print("Skewed input: one key holds 90% of the records.\n")
print(f"{'':14s}{'shuffle write':>17s}{'seconds':>9s}")
print(f"{'groupByKey':14s}{sk_gbk['write_bytes']:>15,} B{sk_gbk_secs:>9.2f}")
print(f"{'reduceByKey':14s}{sk_rbk['write_bytes']:>15,} B{sk_rbk_secs:>9.2f}")
print(f"\n{sk_gbk['write_bytes'] / sk_rbk['write_bytes']:,.0f}x more data shuffled, "
      f"and all of one key's share of it lands on ONE executor.")

Skewed input: one key holds 90% of the records.

                  shuffle write  seconds
groupByKey         72,079,794 B     0.35
reduceByKey            10,365 B     0.21

6,954x more data shuffled, and all of one key's share of it lands on ONE executor.


In [7]:
# The task-duration distribution inside the stage that READS the shuffle.  This is what
# Exercise 10 sends you to the Spark UI to look at: one task doing all the work while
# the others finish at once.
print("Task run time (ms) on the read side of the shuffle\n")
print(f"{'':14s}{'min':>8s}{'median':>9s}{'max':>8s}{'max/median':>13s}")
for label, m in (("groupByKey", sk_gbk), ("reduceByKey", sk_rbk)):
    print(f"{label:14s}{m['task_min']:>8,.0f}{m['task_median']:>9,.0f}"
          f"{m['task_max']:>8,.0f}{m['straggler']:>12.1f}x")

print("\nThe straggler is the max/median ratio.  One task holds the hot key; everything")
print("downstream waits for it, because a shuffle is a barrier.  This is the '99% complete'")
print("pathology of chapter 1, seen from inside the stage.")
print("\nNote what the WALL CLOCK does not show.  At eight million records on a laptop the")
print("hot key's list still fits in memory, so groupByKey finishes -- only slower and more")
print("unevenly.  The failure mode is not gradual: it is fine, fine, fine, and then the")
print("executor holding that one key runs out of memory and the job dies.  The shuffle")
print("size and the task distribution are the two numbers that warn you first, which is")
print("why they are worth reading before the job gets big enough to fail.")

Task run time (ms) on the read side of the shuffle

                   min   median     max   max/median
groupByKey           4        5     122        24.4x
reduceByKey          3        3       4         1.3x

The straggler is the max/median ratio.  One task holds the hot key; everything
downstream waits for it, because a shuffle is a barrier.  This is the '99% complete'
pathology of chapter 1, seen from inside the stage.

Note what the WALL CLOCK does not show.  At eight million records on a laptop the
hot key's list still fits in memory, so groupByKey finishes -- only slower and more
unevenly.  The failure mode is not gradual: it is fine, fine, fine, and then the
executor holding that one key runs out of memory and the job dies.  The shuffle
size and the task distribution are the two numbers that warn you first, which is
why they are worth reading before the job gets big enough to fail.


## 5. `sortByKey`, and the shuffle nobody expects

`sortByKey` is on the chapter's list of wide dependencies, and it is worth its own look
because it costs more than it appears to. Sorting globally means every record must end up in
the right partition, so Spark **range-partitions** the data — and to choose the boundaries it
first has to *sample* the keys, which is an extra job that runs before the sort does.

In [8]:
text = [('What Will It Take for BU Commuters to Leave Their Cars for the MBTA? University '
         'boosts T pass subsidies to cover half the cost, raises parking fees, all part of '
         'broader strategy to build a greener BU')]
words = (sc.parallelize(text)
           .flatMap(lambda x: x.split(' '))
           .filter(lambda w: w)
           .map(lambda w: (w.lower().strip('?,'), 1)))

before = len(_get("/jobs"))
counts = words.reduceByKey(lambda a, b: a + b)
sorted_counts, _ = timed("sortByKey", lambda: counts.sortByKey().collect())
after = len(_get("/jobs"))

print("first 8 in key order:", sorted_counts[:8])
print(f"\njobs submitted by the single sortByKey().collect() : {after - before}")
print("One of them is the sampling job that decides the range boundaries; the other is")
print("the sort itself.  A transformation that quietly submits its own job is unusual,")
print("and it is the reason sortByKey is more expensive than its one line suggests.")

first 8 in key order: [('a', 1), ('all', 1), ('boosts', 1), ('broader', 1), ('bu', 2), ('build', 1), ('cars', 1), ('commuters', 1)]

jobs submitted by the single sortByKey().collect() : 3
One of them is the sampling job that decides the range boundaries; the other is
the sort itself.  A transformation that quietly submits its own job is unusual,
and it is the reason sortByKey is more expensive than its one line suggests.


In [9]:
# If you only want the largest few, do NOT sort the whole data set.
# top(n) keeps n items per partition and merges: no shuffle at all.
print("top(5) by count :", counts.top(5, key=lambda kv: kv[1]))
print("\nsortByKey shuffles everything to answer a question about five records.")

top(5) by count : [('to', 3), ('for', 2), ('bu', 2), ('the', 2), ('cover', 1)]

sortByKey shuffles everything to answer a question about five records.


## 6. Exercise 7: four operators that work and should not be used

In [10]:
kv = sc.parallelize([('a', 1), ('b', 2), ('a', 3), ('c', 4), ('b', 5)], 4)

print("(a) map where only the value changes")
print("    poor   : kv.map(lambda kv: (kv[0], kv[1] * 10))")
print("    better : kv.mapValues(lambda v: v * 10)")
print("    why    : mapValues promises the keys are unchanged, so a partitioning by key")
print("             survives it and a later by-key operation need not shuffle again.")
print("    same?  ", sorted(kv.map(lambda p: (p[0], p[1] * 10)).collect())
      == sorted(kv.mapValues(lambda v: v * 10).collect()))

print("\n(b) groupByKey followed by an average")
poor = kv.groupByKey().mapValues(lambda vs: sum(vs) / len(vs))
better = (kv.aggregateByKey((0.0, 0),
                            lambda acc, v: (acc[0] + v, acc[1] + 1),
                            lambda a, b: (a[0] + b[0], a[1] + b[1]))
            .mapValues(lambda acc: acc[0] / acc[1]))
print("    poor   : kv.groupByKey().mapValues(lambda vs: sum(vs) / len(vs))")
print("    better : kv.aggregateByKey((0.0, 0), add-one, merge-two).mapValues(divide)")
print("    why    : the mean IS incrementally combinable, if the accumulator carries the")
print("             count as well as the sum.  groupByKey holds every value to compute it.")
print("    same?  ", sorted(poor.collect()) == sorted(better.collect()))

print("\n(c) collect() then a Python loop")
print("    poor   : total = sum(v for _, v in kv.collect())")
print("    better : kv.values().sum()")
print("    why    : collect() puts the whole data set on the driver, which is the premise")
print("             distributed processing exists to escape, and then works sequentially.")
print("    same?  ", sum(v for _, v in kv.collect()) == kv.values().sum())

print("\n(d) repartition(10) at the end of a job to write ten files")
print("    poor   : result.repartition(10).saveAsTextFile(...)")
print("    better : result.coalesce(10).saveAsTextFile(...)")
print("    why    : repartition forces a full shuffle to REDUCE the partition count.")
print("             coalesce merges existing partitions and moves nothing over the network.")
print("             Notebook 2.6 shows what each does to the layout.")

(a) map where only the value changes
    poor   : kv.map(lambda kv: (kv[0], kv[1] * 10))
    better : kv.mapValues(lambda v: v * 10)
    why    : mapValues promises the keys are unchanged, so a partitioning by key
             survives it and a later by-key operation need not shuffle again.
    same?   True

(b) groupByKey followed by an average
    poor   : kv.groupByKey().mapValues(lambda vs: sum(vs) / len(vs))
    better : kv.aggregateByKey((0.0, 0), add-one, merge-two).mapValues(divide)
    why    : the mean IS incrementally combinable, if the accumulator carries the
             count as well as the sum.  groupByKey holds every value to compute it.


    same?   True

(c) collect() then a Python loop
    poor   : total = sum(v for _, v in kv.collect())
    better : kv.values().sum()
    why    : collect() puts the whole data set on the driver, which is the premise
             distributed processing exists to escape, and then works sequentially.
    same?   True

(d) repartition(10) at the end of a job to write ten files
    poor   : result.repartition(10).saveAsTextFile(...)
    better : result.coalesce(10).saveAsTextFile(...)
    why    : repartition forces a full shuffle to REDUCE the partition count.
             coalesce merges existing partitions and moves nothing over the network.
             Notebook 2.6 shows what each does to the layout.


## Conclusion

**The rule.** If the grouped values are going to be aggregated anyway, do not use
`groupByKey`. Prefer `reduceByKey`, `aggregateByKey`, or `combineByKey`. `groupByKey` is
appropriate only when the complete collection of values for a key is genuinely required and no
incremental combine is possible — and even then the requirement is worth a second look.

**The measurement**, on eight million records and 50 keys: the same answer, the same number
of records shuffled, and hundreds of times the bytes. The mechanism is the map-side combine,
inherited from Hadoop, and it is the reason `reduceByKey` scales with the number of *keys*
while `groupByKey` scales with the number of *values*.

**The failure**, which is what makes it more than an efficiency note: `groupByKey` materializes
all of one key's values on one executor, so one popular key is enough to fail a job whose data
would otherwise fit. The task-duration distribution in section 4 is what that looks like before
it fails.

**The caveat, from chapter 5.** Switching to `aggregateByKey` or `combineByKey` does not by
itself buy safety. The test to apply before writing any aggregation is whether the *combined
accumulator is smaller than the sum of its parts*. If merging does not shrink the data, the
operation is a regrouping wearing a reduction's clothes, and it carries a regrouping's risks —
which is precisely what `groupByKey` is.

**Next.** Notebook 2.5 takes the three by-key aggregators — `reduceByKey`, `aggregateByKey`
and `combineByKey` — and shows what each is for.